# 3D rail defect detection - metasurface training (main run)

Trains the single-layer metasurface + soft detectors + linear class head on the 3D-simulated
fields (detection margin vs intact + crack/dent/wear classification + robustness terms).

**Prerequisite:** the full dataset exists in `data/generated/` - generate it on the 5090 with
`python generate_dataset_3d.py --profile lab` (under 1 h; resumable). See `SETUP_LAB.md` section 5.

Two runs, per the plan: **(1)** idealized phase-only `SLM2D`, **(2)** real meta-atom
`MetaUnitSoft` (pillar widths through the Face3D library fits, central 60x30 crop).
Both auto-resume from `data/checkpoints/<run_name>/latest.pt` if interrupted (verified
bit-identical in V8) - safe on the shared workstation.

> **Rev.2 objective:** training uses the **rank objective** (pairwise soft-AUC on
> per-sample-normalized barcodes) and reports pass/false-alarm at a threshold
> **calibrated to 5% FPR on the validation intact spread** — the fixed 0.40 margin was
> ill-posed against the (physically required) ±4 mm placement augmentation.
> Legacy behavior: `TrainConfig(objective='margin', metric='l2')`.


In [ ]:
# Setup: run from the rail3D folder with its venv (see SETUP_LAB.md).
# Device resolution: RAIL3D_DEVICE env var wins; otherwise the 'lab' profile picks the
# strongest CUDA card automatically (cuda:auto), so the 5090's index does not matter.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
import numpy as np, torch, matplotlib.pyplot as plt
from dataclasses import replace
from rail3d import config, data3d, losses3d, train3d, optics3d

PROFILE = 'lab'          # switch to 'laptop' only for small tests
device = config.get_device(PROFILE)
config.ensure_dirs()
print('device:', device)

## Run 1 - phase-only SLM

In [ ]:
cfg_slm = train3d.TrainConfig(
    run_name='ms3d_slm_v1', surface='slm', mode='tot',
    n_epoch=1200, batch_size=config.PROFILES[PROFILE].train_batch,
)
hist_slm = train3d.train(cfg_slm, device=device)

In [ ]:
# Learning curves
def plot_history(hist, title):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
    axes[0].plot([t['loss'] for t in hist['train']]); axes[0].set(title='train loss', xlabel='epoch')
    axes[1].plot([v['auc'] for v in hist['val']], label='AUC')
    axes[1].plot([v['pass_rate'] for v in hist['val']], label='pass@cal')
    axes[1].plot([v['false_alarm'] for v in hist['val']], label='false alarm')
    axes[1].plot([v['class_acc'] for v in hist['val']], label='class acc')
    for p in hist['prune_epochs']: axes[1].axvline(p['epoch'], color='gray', lw=0.5, alpha=0.5)
    axes[1].legend(fontsize=8); axes[1].set(title=title + ' - validation (gray: prunes)', xlabel='epoch')
    axes[2].plot([v['gap_mean'] for v in hist['val']], label='defect gap mean')
    axes[2].plot([v['gap_p5'] for v in hist['val']], label='defect gap p5')
    axes[2].plot([v['gap0_mean'] for v in hist['val']], label='intact gap mean')
    axes[2].plot([v['threshold'] for v in hist['val']], 'r--', lw=0.8, label='calibrated thr')
    axes[2].legend(fontsize=8); axes[2].set(title='barcode gaps', xlabel='epoch')
    fig.tight_layout(); return fig
plot_history(hist_slm, 'SLM');

In [ ]:
# Full evaluation on the test split (hard detector powers everywhere)
model_slm, state_slm = train3d.load_trained(cfg_slm, device, 'best')
data = train3d.load_all_data(cfg_slm, device)
ev_slm = train3d.full_evaluation(model_slm, data, cfg_slm)
print('test:', ev_slm['test'])
print('ROC:', {k: v for k, v in ev_slm['roc'].items() if k != 'points'})
print('confusion (rows=true crack/dent/wear):')
print(ev_slm['confusion'])

In [ ]:
# Trained phase map, detector layout on an output intensity, ROC + noise robustness
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
phase = model_slm.layers[0].phase.detach().cpu()
im = axes[0].imshow(np.mod(phase.numpy() + np.pi, 2*np.pi) - np.pi, cmap='twilight')
plt.colorbar(im, ax=axes[0], fraction=0.04); axes[0].set_title('trained phase (rad)')
with torch.no_grad():
    inten = model_slm.propagate(data['fields'][data['test'][:1]]).abs().pow(2)[0].cpu()
axes[1].imshow(inten.numpy(), cmap='inferno'); axes[1].set_title('output intensity (one test sample)')
c = model_slm.detector.centers().detach().cpu(); dw, dh = model_slm.detector.det_size
from matplotlib.patches import Rectangle
for cx, cy in c:  # imshow: row index = x cell, col index = y cell
    j = (cy + config.WY/2) / config.DX - 0.5; i = (cx + config.WX/2) / config.DX - 0.5
    axes[1].add_patch(Rectangle((j - dh/config.DX/2, i - dw/config.DX/2),
                                dh/config.DX, dw/config.DX, fill=False, edgecolor='cyan', lw=1))
fpr, tpr = ev_slm['roc']['points']
axes[2].plot(fpr, tpr); axes[2].plot([0, 1], [0, 1], 'k:', lw=0.5)
axes[2].set(title='ROC  AUC=%.3f  TPR@1%%FPR=%.3f' % (ev_slm['roc']['auc'], ev_slm['roc']['tpr_at_1pct_fpr']),
            xlabel='FPR', ylabel='TPR')
nc = ev_slm['noise_curve']
axes[3].plot([r['sigma'] for r in nc], [r['pass_rate'] for r in nc], 'o-', label='pass rate')
axes[3].plot([r['sigma'] for r in nc], [r['class_acc'] for r in nc], 's-', label='class acc')
axes[3].plot([r['sigma'] for r in nc], [r['false_alarm'] for r in nc], 'x-', label='false alarm')
axes[3].legend(fontsize=8); axes[3].set(title='noise robustness', xlabel='field noise sigma (rel.)')
fig.tight_layout()

In [ ]:
# Alignment robustness + export the phase map (deliverable, like Face3D's phase.csv)
al = ev_slm['alignment_curve']
plt.figure(figsize=(5, 3)); plt.plot([r['shift_mm'] for r in al], [r['pass_rate'] for r in al], 'o-')
plt.xlabel('detector-plane shift x (mm)'); plt.ylabel('pass rate')
plt.title('alignment robustness'); plt.tight_layout()
np.savetxt(config.CHECKPOINT_DIR / cfg_slm.run_name / 'phase.csv', phase.numpy(), delimiter=',')
print('phase.csv exported')

## Run 2 - real meta-atom parameterization (`MetaUnitSoft`)
Trainable pillar-width map through the per-pixel polynomial library fits
(`library_amp_fit.npy` / `library_phase_fit.npy`, central 60x30 crop; sigmoid width
reparameterization keeps gradients alive at the [1, 3.8] mm bounds).

In [ ]:
cfg_mu = replace(cfg_slm, run_name='ms3d_metaunit_v1', surface='metaunit')
hist_mu = train3d.train(cfg_mu, device=device)
plot_history(hist_mu, 'MetaUnit');

In [ ]:
model_mu, _ = train3d.load_trained(cfg_mu, device, 'best')
ev_mu = train3d.full_evaluation(model_mu, data, cfg_mu)
print('test:', ev_mu['test'])
print('ROC:', {k: v for k, v in ev_mu['roc'].items() if k != 'points'})
print(ev_mu['confusion'])
# fabricable deliverable: pillar-width map in mm
w = model_mu.layers[0].w_pillar.detach().cpu()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
im = axes[0].imshow(w.numpy(), cmap='viridis', vmin=1, vmax=3.8)
plt.colorbar(im, ax=axes[0], fraction=0.04); axes[0].set_title('pillar width (mm)')
im = axes[1].imshow(model_mu.layers[0].phase().detach().cpu().numpy(), cmap='twilight')
plt.colorbar(im, ax=axes[1], fraction=0.04); axes[1].set_title('implied phase (rad)')
im = axes[2].imshow(model_mu.layers[0].amp().detach().cpu().numpy(), cmap='viridis', vmin=0.5, vmax=1)
plt.colorbar(im, ax=axes[2], fraction=0.04); axes[2].set_title('implied amplitude')
fig.tight_layout()
np.savetxt(config.CHECKPOINT_DIR / cfg_mu.run_name / 'w_pillar.csv', w.numpy(), delimiter=',')
print('w_pillar.csv exported')

## Notes
- **Two-layer variant** (config change only): `replace(cfg_slm, run_name='ms3d_slm_2layer',
  n_layer=2, layer_distances=(80.0, 160.0))` - first number = MS-to-MS spacing (adjustable),
  last = final MS-to-detector distance.
- Field mode `'sca'` (psi1 only) trains on scattering alone; `'tot'` (psi0+psi1+psi2, default)
  is what a real measurement sees.
- Compare against the no-metasurface baseline: `training_3d_no_ms_notebook.ipynb`.